In [2]:
import networkx as nx # Used for creating, manipulating, and studying the structure, dynamics, and functions of complex networks.
import matplotlib.pyplot as plt # Used for creating static, animated, and interactive visualizations in Python.
import numpy as np


In [3]:
def create_initial_graph(n_nodes):
    """Creates the basic graph G with n_nodes.
       Args:
          n_nodes (int): The number of nodes in the graph.

       Returns:
          networkx.Graph: The initialized graph G.
    """
    G = nx.Graph()
    # Add nodes to the graph from 0 to n_nodes-1
    G.add_nodes_from([i for i in range(n_nodes)])
    edge_list = []

    # This loop structure creates two cliques of 3 nodes each
    # For i=0, it adds edges for nodes (0,1), (0,2), (1,2) forming a clique of (0,1,2).
    # For i=1, it adds edges for nodes (3,4), (3,5), (4,5) forming a clique of (3,4,5).
    for i in range(2):
        for j in range(3):
            for l in range(j, 3):
                if j != l:
                    # Add edge with a weight of 1.0
                    edge_list.append((i * 3 + j, i * 3 + l, {'weight': 1.0}))

    # Add an additional edge connecting the two cliques to form a single graph
    edge_list = edge_list + [
        (1, 4, {'weight': 1.0}),
    ]
    G.add_edges_from(edge_list)
    print(f"Initial Basic Graph G created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
    return G

Create benchmark graph — y complete graphs of x nodes, connected in a chain

In [4]:
def create_clique_chain_graph(x, y):
    """Creates a graph consisting of y complete graphs (cliques), each with x nodes,
    connected in a cycle — each clique is joined to the next by a single edge,
    and the last clique wraps back to the first.

    Node numbering is contiguous: clique k contains nodes [k*x, ..., k*x + x - 1].
    The bridge between clique k and clique (k+1) % y connects the last node of
    clique k to the first node of clique (k+1) % y.

    Args:
        x (int): Number of nodes in each complete graph (clique size).
        y (int): Number of complete graphs (cliques).

    Returns:
        networkx.Graph: The constructed graph.

    Example:
        x=3, y=3  ->  three triangles (0-1-2), (3-4-5), (6-7-8)
                       bridged by edges (2,3), (5,6), (8,0)
    """
    G = nx.Graph()
    G.add_nodes_from(range(x * y))

    for k in range(y):
        base = k * x

        # Add all edges within clique k (complete subgraph)
        for i in range(x):
            for j in range(i + 1, x):
                G.add_edge(base + i, base + j, weight=1.0)

        # Bridge: last node of clique k -> first node of clique (k+1) % y
        next_base = ((k + 1) % y) * x
        G.add_edge(base + x - 1, next_base, weight=1.0)

        N = G.number_of_nodes()

    print(f"Graph created: {y} cliques of size {x} in a cycle  |  "
          f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    return G, N

In [ ]:
# Get standard benchmark graphs from NetworkX

def get_les_miserables_graph():
    """Returns the Les Misérables co-appearance graph.

    A weighted undirected graph in which nodes represent characters
    from the novel and edges represent co-appearances in the same
    chapter. Commonly used as a benchmark for community detection.

    Returns:
        tuple: A tuple containing:
            - G (networkx.Graph): The Les Misérables co-appearance graph.
            - N (int): The number of nodes in the graph.
    """
    G = nx.les_miserables_graph() # Load the built-in Les Misérables graph from NetworkX
    N = G.number_of_nodes() # Total number of character nodes in the graph
    return G, N


def get_karate_club_graph():
    """Returns Zachary's Karate Club graph.

    A well-known social network graph representing friendships between
    34 members of a karate club. Commonly used as a benchmark for
    community detection, as the club split into two factions.

    Returns:
        tuple: A tuple containing:
            - G (networkx.Graph): The Karate Club graph.
            - N (int): The number of nodes in the graph.
    """
    G = nx.karate_club_graph() # Load the built-in Karate Club graph from NetworkX
    N = G.number_of_nodes() # Total number of member nodes in the graph
    return G, N


def get_davis_southern_women_graph():
    """Returns the Davis Southern Women bipartite graph.

    A bipartite graph representing the attendance of 18 women at
    14 social events in the American South during the 1930s.
    Commonly used as a benchmark for bipartite community detection.

    Returns:
        tuple: A tuple containing:
            - G (networkx.Graph): The Davis Southern Women graph.
            - N (int): The number of nodes in the graph.
    """
    G = nx.davis_southern_women_graph() # Load the built-in Davis Southern Women graph from NetworkX
    N = G.number_of_nodes() # Total number of nodes (women + events) in the graph
    return G, N

2. Create modularity matrix

In [6]:
def create_modularity_matrix(G, N):
    """Calculates the modularity matrix B for a given graph G.

    The modularity matrix is a key component in community detection algorithms
    like the Newman-Girvan algorithm, quantifying how much denser connections
    within communities are compared to connections between communities.

    Args:
        G (networkx.Graph): The input graph for which to calculate the modularity matrix.
        N (int): The number of nodes in the original graph G.
        
    Returns:
        numpy.ndarray: The modularity matrix B.
    """
    # Ensure a consistent order for nodes when converting to numpy array and calculating degrees.
    nodes = list(G.nodes())
    node_to_idx = {node: i for i, node in enumerate(nodes)}

    # A_ij: Adjacency matrix (weight of edge between i and j). For unweighted graphs, it's 1 if connected, 0 otherwise.
    adjacency_matrix = nx.to_numpy_array(G, nodelist=nodes)

    # k_i: Degree of node i. For unweighted graphs, this is the number of edges connected to node i.
    node_degrees = np.array([G.degree(node) for node in nodes])

    # 2m: Sum of all node degrees, which is twice the number of edges in the graph.
    two_m = np.sum(node_degrees)

    # Initialize the Modularity Matrix B with zeros.
    modularity_matrix = np.zeros((N, N))

    # Calculate B_ij = A_ij - (k_i * k_j) / (2m)
    # This formula compares the actual number of edges (A_ij) with the expected number
    # of edges in a random graph with the same degree distribution.
    for i in range(N):
        for j in range(N):
            if two_m == 0: # Avoid division by zero for graphs with no edges
                modularity_matrix[i, j] = adjacency_matrix[i, j]
            else:
                modularity_matrix[i, j] = adjacency_matrix[i, j] - (node_degrees[i] * node_degrees[j]) / two_m


    print("\n\nModularity Matrix for Graph G:")
    print(modularity_matrix)
    return modularity_matrix

3. Create supernode structure

In [7]:
def create_supernode(G, K):
    """Creates a one-hot encoded supernode graph from an initial graph G.

    For one-hot encoding, each node in the original graph G gets 'k' copies,
    where 'k' is the number of communities. These copies represent potential
    community assignments for the original node.

    Args:
        G (networkx.Graph): The original graph.
        k (int): The number of communities, which determines the number of copies
                (qubits) for each original node.

    Returns:
        networkx.Graph: A new graph with supernode structure, where nodes are
                        labeled as (original_node_id, community_index).
    """
    # K is defined in a previous cell as the number of communities.
    # For one-hot encoding, each node gets 'K' copies, where 'K' is the number of communities.
    num_copies_per_node = K # Using the K defined in the notebook

    # Create a new graph for the one-hot supernode structure
    G_onehot_supernodes_labeled = nx.Graph()

    # Add supernodes and intra-supernode connections
    # Each original node u gets num_copies_per_node copies: (u, 0), ..., (u, num_copies_per_node-1)
    # These copies form a clique within each supernode to enforce the one-hot constraint
    # (i.e., a node can only be in one community).
    for u in G.nodes():
        for i in range(num_copies_per_node):
            for j in range(i + 1, num_copies_per_node):
                G_onehot_supernodes_labeled.add_edge((u, i), (u, j))

    # Add inter-supernode connections based on original graph edges
    # If an edge exists between u and v in the original graph, then connections are added
    # between their corresponding 'community slots' in the supernode graph. This means
    # if u and v are both assigned to community 'i', there's an edge between (u,i) and (v,i).
    for u, v in G.edges():
        for i in range(num_copies_per_node):
            G_onehot_supernodes_labeled.add_edge((u, i), (v, i))

    print(f"\n\nOne-Hot Supernode Graph (Labeled) - Number of nodes: {G_onehot_supernodes_labeled.number_of_nodes()}")
    print(f"One-Hot Supernode Graph (Labeled) - Number of edges: {G_onehot_supernodes_labeled.number_of_edges()}")

    return G_onehot_supernodes_labeled

4. Utility functions

In [8]:
def display_graph(graph_input, title="Graph Visualization", node_colors='skyblue', seed=700):
    """Displays a NetworkX graph.

    Args:
        graph_input (networkx.Graph): The graph to be displayed.
        title (str, optional): The title of the graph visualization. Defaults to "Graph Visualization".
        node_colors (str or list, optional): The color(s) for the nodes. Can be a single color string
                                            or a list of colors, one for each node. Defaults to 'skyblue'.
        seed (int, optional): Seed for the random number generator used in `spring_layout`
                              to ensure consistent layout. Defaults to 700.
    """
    plt.figure(figsize=(8, 6)) # Set the figure size for better visualization
    # Generate node positions using spring_layout for a visually appealing layout.
    # 'seed' ensures reproducibility of the layout.
    # 'k' parameter specifies optimal distance between nodes.
    pos = nx.spring_layout(graph_input, seed=seed, k=0.3)
    # Draw the graph with specified positions, labels, colors, sizes, and edge colors.
    nx.draw_networkx(graph_input, pos, with_labels=True, node_color=node_colors, node_size=700, edge_color='gray', font_size=10)
    plt.title(title) # Set the title of the plot
    plt.axis('off') # Turn off the axis to display only the graph
    plt.show() # Display the plot

In [9]:
def spectral_stop(B):
    """Checks the spectral stopping criterion for graph partitioning.

    Computes the largest eigenvalue of the modularity matrix B. If it is
    non-positive, no further community structure can be extracted and
    the algorithm should stop.

    Idea described in:
    Newman, M. E. J. (2006). "Modularity and community structure in networks."
    Proceedings of the National Academy of Sciences, 103(23), 8577-8582.
    DOI: 10.1073/pnas.0601602103 | arXiv: physics/0602124

    Args:
        B (numpy.ndarray): The modularity matrix of the (sub)graph.

    Returns:
        bool: True if the maximum eigenvalue <= 0 (stop), False otherwise (continue).
    """
    # Compute all eigenvalues of B; take the real part to discard numerical noise
    eigvals = np.linalg.eigvals(B)
    lambda_max = np.max(eigvals.real)
    print(f"\n\nMaximum eigenvalue of the modularity matrix: {lambda_max}")

    # A non-positive maximum eigenvalue means splitting this (sub)graph
    # would not improve modularity, so signal the algorithm to stop
    if lambda_max <= 0:
        return True
    else:
        return False